# VaultBox Colab Server
Run this notebook to connect Google Colab as a temporary transfer worker for VaultBox.

Colab does not manage provider accounts. Copy the server URL and token into VaultBox, then control transfers from the app.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq p7zip-full unrar git curl
!python -m pip install -q --upgrade pip


In [ ]:
import os, shutil, subprocess, pathlib
REPO = os.environ.get('VAULTBOX_PUBLIC_REPO', 'https://github.com/CeatursHarmginton/vault-box.git')
ROOT = pathlib.Path('/content/vaultbox-public')
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.check_call(['git', 'clone', '--depth', '1', REPO, str(ROOT)])
os.chdir(ROOT / 'colab')
!python -m pip install -q -r requirements.txt
print('Colab source ready:', pathlib.Path.cwd())


In [ ]:
import os, secrets, subprocess, sys, time
from google.colab import output
os.environ['COLAB_SERVER_TOKEN'] = os.environ.get('COLAB_SERVER_TOKEN') or secrets.token_urlsafe(32)
server_url = output.eval_js('google.colab.kernel.proxyPort(8000)')
os.environ['COLAB_PUBLIC_URL'] = server_url
proc = subprocess.Popen([sys.executable, 'scripts/start_colab_server.py'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(3)
print('Server URL:', server_url)
print('Connection Token:', os.environ['COLAB_SERVER_TOKEN'])
print('Copy these into VaultBox app -> Connect Colab')


In [ ]:
import time
print('VaultBox Colab worker is running. Keep this cell active.')
while True:
    line = proc.stdout.readline() if proc.stdout else ''
    if line:
        print(line.rstrip())
    time.sleep(1)
